# Notebook 04: RAG Data Preparation

## Objetivo
Transformar pseudo-labels + transcripcion bruta en documentos indexables para RAG con dos capas:

| Capa | Contenido | Uso |
|------|-----------|-----|
| **Enriquecida** | topic, explanation, bullets, example, keywords | Indice principal |
| **Evidencia** | Transcripcion + OCR original | Verificacion / trazabilidad |

## Pipeline
```
raw_train.jsonl + final_train.jsonl
    ↓
1. Merge pseudo-labels con datos raw
2. Correccion de terminos canonicos
3. Asignacion de quality_score (0-1)
4. Clasificacion: docs / review / rejected
5. Crear documentos enriquecidos + evidencia
    ↓
rag_docs.jsonl       (buenos → indice principal)
rag_evidence.jsonl   (evidencia bruta enlazada)
rag_review.jsonl     (dudosos → revision manual)
rag_rejected.jsonl   (basura → fuera)
```

## Inputs
- `data/raw_train.jsonl` — dataset instruct sin labels
- `data/raw_val.jsonl` — validation set
- `data/final_train.jsonl` / `data/checkpoint_train.jsonl` — pseudo-labels generadas
- `artifacts/audio.json` — transcripcion Whisper completa

---
## Setup

In [ ]:
import json
import re
import hashlib
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from collections import Counter
from copy import deepcopy
from dataclasses import dataclass, field, asdict

# Paths — detect Proyecto_Final_Alejandro/notebooks/ and go up 2 levels
cwd = Path.cwd()
if cwd.name == "notebooks" and cwd.parent.name == "Proyecto_Final_Alejandro":
    PROJECT_ROOT = cwd.parent.parent
elif cwd.name in ("notebooks", "Proyecto_Final_Alejandro"):
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RAG_DIR = PROJECT_ROOT / "artifacts" / "rag"
RAG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"RAG output:   {RAG_DIR}")

---
## 1. Cargar datos

In [ ]:
def load_jsonl(path: Path) -> List[Dict]:
    """Carga JSONL."""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data


def save_jsonl(data: List[Dict], path: Path):
    """Guarda JSONL."""
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


# Cargar pseudo-labels (usar checkpoint si tiene más datos que final)
final_path = DATA_DIR / "final_train.jsonl"
checkpoint_path = DATA_DIR / "checkpoint_train.jsonl"
val_path = DATA_DIR / "final_val.jsonl"

# Elegir el archivo con más datos
if checkpoint_path.exists() and final_path.exists():
    final_data = load_jsonl(final_path)
    checkpoint_data = load_jsonl(checkpoint_path)
    labeled_data = checkpoint_data if len(checkpoint_data) > len(final_data) else final_data
    print(f"Usando {'checkpoint' if len(checkpoint_data) > len(final_data) else 'final'}: {len(labeled_data)} ejemplos")
elif checkpoint_path.exists():
    labeled_data = load_jsonl(checkpoint_path)
elif final_path.exists():
    labeled_data = load_jsonl(final_path)
else:
    raise FileNotFoundError("No se encontró final_train.jsonl ni checkpoint_train.jsonl")

# Validation
if val_path.exists():
    val_data = load_jsonl(val_path)
    labeled_data.extend(val_data)
    print(f"+ Validation: {len(val_data)} ejemplos")

# Raw data (para evidencia)
raw_train = load_jsonl(DATA_DIR / "raw_train.jsonl") if (DATA_DIR / "raw_train.jsonl").exists() else []
raw_val = load_jsonl(DATA_DIR / "raw_val.jsonl") if (DATA_DIR / "raw_val.jsonl").exists() else []
raw_data = {ex["id"]: ex for ex in raw_train + raw_val}

# Audio transcription
audio_path = ARTIFACTS_DIR / "audio.json"
audio_data = {}
if audio_path.exists():
    with open(audio_path) as f:
        audio_data = json.load(f).get("audio", {})
    print(f"Audio: {len(audio_data.get('segments', []))} segmentos, {audio_data.get('duration_ms', 0)/1000:.0f}s")

print(f"\nTotal con pseudo-labels: {len(labeled_data)} ejemplos")
print(f"Total raw (para evidencia): {len(raw_data)} ejemplos")

---
## 2. Extraer pseudo-labels de los mensajes

In [ ]:
def extract_json(content: str) -> str:
    """Extrae JSON de un string (elimina fences markdown)."""
    content = re.sub(r'^```(?:json)?\s*\n?', '', content.strip(), flags=re.MULTILINE)
    content = re.sub(r'\n?```\s*$', '', content)
    m = re.search(r'\{[\s\S]*\}', content)
    return m.group(0) if m else content


def extract_pseudolabel(example: Dict) -> Optional[Dict]:
    """Extrae el pseudo-label del mensaje assistant."""
    for msg in example.get("messages", []):
        if msg["role"] == "assistant" and msg["content"].strip():
            try:
                return json.loads(extract_json(msg["content"]))
            except (json.JSONDecodeError, Exception):
                return None
    return None


def extract_user_content(example: Dict) -> str:
    """Extrae el contenido del mensaje user."""
    for msg in example.get("messages", []):
        if msg["role"] == "user":
            return msg["content"]
    return ""


def extract_transcript(user_content: str) -> str:
    """Extrae la transcripción del contenido user."""
    m = re.search(r'Transcripción:\s*(.+?)(?:\n---|\Z)', user_content, re.DOTALL)
    if m:
        return m.group(1).strip()
    return user_content[:800]


# Extraer todos los pseudo-labels
records = []
parse_errors = 0

for ex in labeled_data:
    label = extract_pseudolabel(ex)
    if label is None:
        parse_errors += 1
        continue
    
    user_content = extract_user_content(ex)
    transcript = extract_transcript(user_content)
    
    records.append({
        "id": ex["id"],
        "meta": ex.get("meta", {}),
        "label": label,
        "user_content": user_content,
        "transcript": transcript,
    })

print(f"Pseudo-labels extraídos: {len(records)}")
if parse_errors:
    print(f"Errores de parseo: {parse_errors}")

---
## 3. Corrección de términos canónicos

El ASR produce errores frecuentes en términos técnicos. Corregimos los más comunes.

In [ ]:
# Diccionario de correcciones canónicas
# Formato: {"error_común": "término_correcto"}
CANONICAL_TERMS = {
    # ML/DL terms
    "back of works": "Bag of Words",
    "bag of works": "Bag of Words",
    "back of words": "Bag of Words",
    "bac of words": "Bag of Words",
    "bags of words": "Bag of Words",
    "redes neuronales": "redes neuronales",
    "neur network": "neural network",
    "neuronal network": "neural network",
    "learning rate": "learning rate",
    "learnin rate": "learning rate",
    "overfiting": "overfitting",
    "over fitting": "overfitting",
    "over-fitting": "overfitting",
    "underfiting": "underfitting",
    "under fitting": "underfitting",
    "under-fitting": "underfitting",
    "gradient decent": "gradient descent",
    "gradien descent": "gradient descent",
    "backpropagation": "backpropagation",
    "back propagation": "backpropagation",
    "convolution": "convolución",
    "data aumentation": "data augmentation",
    "data augmentación": "data augmentation",
    "fine tunning": "fine-tuning",
    "fine tuning": "fine-tuning",
    "fine tunnig": "fine-tuning",
    "transfer lerning": "transfer learning",
    "transferlearning": "transfer learning",
    "cross validación": "cross-validation",
    "cross validation": "cross-validation",
    "hiperparámetro": "hiperparámetro",
    "hyperparametro": "hiperparámetro",
    "embedings": "embeddings",
    "embeding": "embedding",
    "tokenizacion": "tokenización",
    # Frameworks
    "tensorflow": "TensorFlow",
    "pytorch": "PyTorch",
    "pytoch": "PyTorch",
    "scikit learn": "scikit-learn",
    "sklearn": "scikit-learn",
    # Architectures
    "resnet": "ResNet",
    "alexnet": "AlexNet",
    "bert": "BERT",
    "gpt": "GPT",
    "transformer": "Transformer",
    "transformers": "Transformers",
    "attention": "attention",
    "self attention": "self-attention",
    "self-atention": "self-attention",
    # Stats/Math
    "pca": "PCA",
    "svd": "SVD",
    "umap": "UMAP",
    "tsne": "t-SNE",
    "t-sne": "t-SNE",
    "knn": "KNN",
    "k nearest": "K-nearest neighbors",
    "random forest": "Random Forest",
    "decision tree": "Decision Tree",
    # NLP
    "nlp": "NLP",
    "natural language processing": "NLP",
    "procesamiento lenguaje natural": "NLP",
    "tfidf": "TF-IDF",
    "tf-idf": "TF-IDF",
    "tf idf": "TF-IDF",
    "word2vec": "Word2Vec",
    "word 2 vec": "Word2Vec",
    # Common ASR errors in Spanish tech context
    "datos estupurados": "datos estructurados",
    "datos estrupturados": "datos estructurados",
    "descalaridad": "escalabilidad",
    "más net": "ImageNet",
    "image net": "ImageNet",
    "imagenet": "ImageNet",
}


def apply_canonical_corrections(text: str) -> Tuple[str, List[str]]:
    """Aplica correcciones canónicas. Retorna (texto_corregido, [correcciones_aplicadas])."""
    corrections = []
    corrected = text
    
    for error, canonical in CANONICAL_TERMS.items():
        pattern = re.compile(re.escape(error), re.IGNORECASE)
        if pattern.search(corrected):
            corrected = pattern.sub(canonical, corrected)
            corrections.append(f"{error} → {canonical}")
    
    return corrected, corrections


# Aplicar correcciones a todos los records
total_corrections = 0
correction_log = []

for rec in records:
    label = rec["label"]
    all_corrections = []
    
    # Corregir campos de texto
    for field_name in ["topic", "explanation", "example"]:
        if field_name in label and isinstance(label[field_name], str):
            corrected, corrs = apply_canonical_corrections(label[field_name])
            label[field_name] = corrected
            all_corrections.extend(corrs)
    
    # Corregir bullets
    if "bullets" in label and isinstance(label["bullets"], list):
        for i, bullet in enumerate(label["bullets"]):
            corrected, corrs = apply_canonical_corrections(bullet)
            label["bullets"][i] = corrected
            all_corrections.extend(corrs)
    
    # Corregir keywords
    if "keywords" in label and isinstance(label["keywords"], list):
        for i, kw in enumerate(label["keywords"]):
            corrected, corrs = apply_canonical_corrections(kw)
            label["keywords"][i] = corrected
            all_corrections.extend(corrs)
    
    if all_corrections:
        total_corrections += len(all_corrections)
        correction_log.append({"id": rec["id"], "corrections": all_corrections})

print(f"Correcciones aplicadas: {total_corrections} en {len(correction_log)} documentos")
if correction_log[:5]:
    print("\nEjemplos:")
    for entry in correction_log[:5]:
        print(f"  {entry['id']}: {entry['corrections']}")

---
## 4. Quality Scoring

Cada documento recibe un `quality_score` (0-1) basado en reglas deterministas.

| Criterio | Peso | Detalle |
|----------|------|---------|
| Completitud de campos | 0.20 | Todos los campos requeridos presentes |
| Keywords coherentes | 0.20 | 4+ keywords, sin junk |
| Bullets informativos | 0.20 | 5 bullets, >10 chars, sin repetición |
| Explanation sustancial | 0.15 | >30 chars, no genérica |
| Evidence presente | 0.10 | source.evidence no vacío |
| Topic concreto | 0.10 | 2-5 palabras, sin metadatos |
| Timestamp válido | 0.05 | start < end, rango razonable |

In [ ]:
METADATA_RE = re.compile(
    r'\b(SLIDE_OCR|SUBTITLES_OCR|TEACHER_AUDIO|TIME_RANGE|SLIDE_ID|'
    r'frame_\d*|Genera los apuntes|genera los apuntes)\b|'
    r'\b\d+ms\b',
    re.IGNORECASE
)

JUNK_KEYWORDS = {
    'slide', 'frame', 'time', 'range', 'ocr', 'audio', 'teacher',
    'genera', 'apuntes', 'json', 'input', 'output', 'subtítulos',
    'transcripción'
}

GENERIC_EXPLANATIONS = {
    "consulta la grabación",
    "ver la grabación",
    "ver el vídeo",
    "revisa el material",
}


def compute_quality_score(label: Dict, transcript: str = "") -> Tuple[float, Dict[str, float]]:
    """Calcula quality_score (0-1) con desglose por criterio."""
    scores = {}
    
    # 1. Completitud (0.20)
    required = {'topic', 'explanation', 'bullets', 'example', 'keywords', 'timestamps_ms', 'source'}
    present = required & set(label.keys())
    scores["completeness"] = len(present) / len(required) * 0.20
    
    # 2. Keywords (0.20)
    keywords = label.get("keywords", [])
    if isinstance(keywords, list):
        clean_kw = [kw for kw in keywords if kw.lower() not in JUNK_KEYWORDS and len(kw) > 2]
        kw_score = min(len(clean_kw) / 4, 1.0)  # 4+ keywords = max
    else:
        kw_score = 0
    scores["keywords"] = kw_score * 0.20
    
    # 3. Bullets (0.20)
    bullets = label.get("bullets", [])
    if isinstance(bullets, list) and len(bullets) == 5:
        informative = [b for b in bullets 
                      if isinstance(b, str) 
                      and len(b.strip()) > 10 
                      and not METADATA_RE.search(b)
                      and not any(g in b.lower() for g in GENERIC_EXPLANATIONS)]
        # Check for repetition
        unique_starts = set(b[:30].lower() for b in informative)
        bullet_score = min(len(unique_starts) / 5, 1.0)
    else:
        bullet_score = 0
    scores["bullets"] = bullet_score * 0.20
    
    # 4. Explanation (0.15)
    explanation = label.get("explanation", "")
    if isinstance(explanation, str) and len(explanation) > 30:
        is_generic = any(g in explanation.lower() for g in GENERIC_EXPLANATIONS)
        has_metadata = bool(METADATA_RE.search(explanation))
        exp_score = 0.0 if (is_generic or has_metadata) else 1.0
    else:
        exp_score = 0
    scores["explanation"] = exp_score * 0.15
    
    # 5. Evidence (0.10)
    source = label.get("source", {})
    evidence = source.get("evidence", "") if isinstance(source, dict) else ""
    has_evidence = isinstance(evidence, str) and len(evidence.strip()) > 10
    scores["evidence"] = (1.0 if has_evidence else 0.0) * 0.10
    
    # 6. Topic (0.10)
    topic = label.get("topic", "")
    if isinstance(topic, str):
        topic_words = len(topic.split())
        good_topic = (2 <= topic_words <= 6 
                     and not METADATA_RE.search(topic) 
                     and topic.lower() not in {"segmento de clase", "clase", "tema"})
    else:
        good_topic = False
    scores["topic"] = (1.0 if good_topic else 0.3) * 0.10
    
    # 7. Timestamps (0.05)
    ts = label.get("timestamps_ms", [])
    if isinstance(ts, list) and len(ts) == 2:
        try:
            valid_ts = int(ts[0]) < int(ts[1]) and int(ts[1]) - int(ts[0]) < 600000  # < 10min
        except (TypeError, ValueError):
            valid_ts = False
    else:
        valid_ts = False
    scores["timestamps"] = (1.0 if valid_ts else 0.0) * 0.05
    
    total = sum(scores.values())
    return round(total, 3), scores


# Calcular scores
for rec in records:
    score, breakdown = compute_quality_score(rec["label"], rec["transcript"])
    rec["quality_score"] = score
    rec["quality_breakdown"] = breakdown

# Distribución
scores = [r["quality_score"] for r in records]
print(f"Quality Score Distribution (n={len(scores)}):")
print(f"  Min:    {min(scores):.3f}")
print(f"  Max:    {max(scores):.3f}")
print(f"  Mean:   {sum(scores)/len(scores):.3f}")
print(f"  Median: {sorted(scores)[len(scores)//2]:.3f}")

# Buckets
buckets = {"excellent (>0.8)": 0, "good (0.6-0.8)": 0, "fair (0.4-0.6)": 0, "poor (<0.4)": 0}
for s in scores:
    if s > 0.8: buckets["excellent (>0.8)"] += 1
    elif s > 0.6: buckets["good (0.6-0.8)"] += 1
    elif s > 0.4: buckets["fair (0.4-0.6)"] += 1
    else: buckets["poor (<0.4)"] += 1

print(f"\nBuckets:")
for bucket, count in buckets.items():
    bar = '█' * (count * 40 // max(len(scores), 1))
    print(f"  {bucket:20s}: {count:4d} {bar}")

In [ ]:
# Inspeccionar peores y mejores
sorted_records = sorted(records, key=lambda r: r["quality_score"])

print("=" * 70)
print("PEORES 3 (candidates for rejection):")
print("=" * 70)
for rec in sorted_records[:3]:
    print(f"\n  ID: {rec['id']} | Score: {rec['quality_score']:.3f}")
    print(f"  Topic: {rec['label'].get('topic', 'N/A')}")
    print(f"  Breakdown: {rec['quality_breakdown']}")

print(f"\n{'=' * 70}")
print("MEJORES 3:")
print("=" * 70)
for rec in sorted_records[-3:]:
    print(f"\n  ID: {rec['id']} | Score: {rec['quality_score']:.3f}")
    print(f"  Topic: {rec['label'].get('topic', 'N/A')}")
    print(f"  Keywords: {rec['label'].get('keywords', [])}")

---
## 5. Clasificación: docs / review / rejected

| Categoría | Criterio | Destino |
|-----------|----------|---------|
| **docs** | quality_score >= 0.55 | Índice principal |
| **review** | 0.35 <= quality_score < 0.55 | Revisión manual |
| **rejected** | quality_score < 0.35 | Descartado |

In [ ]:
THRESHOLD_GOOD = 0.55
THRESHOLD_REVIEW = 0.35

docs = []
review = []
rejected = []

for rec in records:
    if rec["quality_score"] >= THRESHOLD_GOOD:
        docs.append(rec)
    elif rec["quality_score"] >= THRESHOLD_REVIEW:
        review.append(rec)
    else:
        rejected.append(rec)

print(f"Clasificación:")
print(f"  docs (indexar):     {len(docs):4d} ({len(docs)/len(records)*100:.1f}%)")
print(f"  review (revisar):   {len(review):4d} ({len(review)/len(records)*100:.1f}%)")
print(f"  rejected (basura):  {len(rejected):4d} ({len(rejected)/len(records)*100:.1f}%)")

---
## 6. Crear documentos para RAG

Cada pseudo-label se convierte en **dos documentos**:

1. **Documento enriquecido** (`source_type: pseudo_label`) — para el índice principal
2. **Documento evidencia** (`source_type: raw_evidence`) — para verificación

In [ ]:
@dataclass
class RAGDocument:
    """Documento indexable para RAG."""
    doc_id: str
    page_content: str
    source_type: str  # "pseudo_label" or "raw_evidence"
    topic: str = ""
    keywords: List[str] = field(default_factory=list)
    slide_id: str = ""
    session_id: str = ""
    start_ms: int = 0
    end_ms: int = 0
    quality_score: float = 0.0
    semantic_risk: str = "low"  # low / medium / high
    needs_review: bool = False
    linked_doc_id: str = ""  # Link enriched <-> evidence
    
    def to_dict(self) -> Dict:
        return asdict(self)


def build_enriched_content(label: Dict) -> str:
    """Construye page_content bien formateado para indexación."""
    parts = []
    
    topic = label.get("topic", "Sin tema")
    parts.append(f"Tema: {topic}")
    
    explanation = label.get("explanation", "")
    if explanation:
        parts.append(f"\nExplicación:\n{explanation}")
    
    bullets = label.get("bullets", [])
    if bullets:
        parts.append("\nPuntos clave:")
        for b in bullets:
            parts.append(f"- {b}")
    
    example = label.get("example", "")
    if example and "grabación" not in example.lower():
        parts.append(f"\nEjemplo:\n{example}")
    
    formula = label.get("formula")
    if formula and isinstance(formula, str):
        parts.append(f"\nFórmula: {formula}")
    
    keywords = label.get("keywords", [])
    if keywords:
        parts.append(f"\nPalabras clave: {', '.join(keywords)}")
    
    return "\n".join(parts)


def classify_risk(quality_score: float, label: Dict) -> str:
    """Clasifica el riesgo semántico del documento."""
    if quality_score >= 0.75:
        return "low"
    elif quality_score >= 0.55:
        # Check for suspicious patterns
        text = json.dumps(label, ensure_ascii=False).lower()
        suspicious = any(term in text for term in ["consulta la grabación", "segmento de clase"])
        return "medium" if suspicious else "low"
    else:
        return "high"


def create_rag_documents(rec: Dict) -> Tuple[RAGDocument, RAGDocument]:
    """Crea par de documentos (enriquecido + evidencia) a partir de un record."""
    label = rec["label"]
    meta = rec["meta"]
    
    # IDs
    enriched_id = f"enriched_{rec['id']}"
    evidence_id = f"evidence_{rec['id']}"
    
    # Timestamps
    ts = label.get("timestamps_ms", [0, 0])
    start_ms = int(ts[0]) if len(ts) >= 1 else meta.get("timestamp_ms", 0)
    end_ms = int(ts[1]) if len(ts) >= 2 else start_ms + 30000
    
    # Common metadata
    slide_id = label.get("source", {}).get("slide_id", meta.get("slide_id", ""))
    session_id = meta.get("session_id", "")
    quality = rec["quality_score"]
    risk = classify_risk(quality, label)
    
    # Documento enriquecido
    enriched = RAGDocument(
        doc_id=enriched_id,
        page_content=build_enriched_content(label),
        source_type="pseudo_label",
        topic=label.get("topic", ""),
        keywords=label.get("keywords", []),
        slide_id=slide_id,
        session_id=session_id,
        start_ms=start_ms,
        end_ms=end_ms,
        quality_score=quality,
        semantic_risk=risk,
        needs_review=(quality < THRESHOLD_GOOD),
        linked_doc_id=evidence_id,
    )
    
    # Documento evidencia
    evidence_content = rec["transcript"]
    if not evidence_content or len(evidence_content) < 20:
        evidence_content = rec["user_content"]
    
    evidence = RAGDocument(
        doc_id=evidence_id,
        page_content=evidence_content,
        source_type="raw_evidence",
        topic=label.get("topic", ""),
        keywords=[],
        slide_id=slide_id,
        session_id=session_id,
        start_ms=start_ms,
        end_ms=end_ms,
        quality_score=quality,
        semantic_risk=risk,
        needs_review=False,
        linked_doc_id=enriched_id,
    )
    
    return enriched, evidence


# Crear documentos
rag_docs = []      # Enriquecidos buenos
rag_evidence = []   # Evidencia bruta enlazada
rag_review_docs = []
rag_rejected_docs = []

for rec in docs:
    enriched, evidence = create_rag_documents(rec)
    rag_docs.append(enriched)
    rag_evidence.append(evidence)

for rec in review:
    enriched, evidence = create_rag_documents(rec)
    rag_review_docs.append(enriched)
    rag_evidence.append(evidence)  # Evidence always gets indexed

for rec in rejected:
    _, evidence = create_rag_documents(rec)
    rag_rejected_docs.append(evidence)  # Only save for audit

print(f"Documentos RAG creados:")
print(f"  Enriquecidos (principal): {len(rag_docs)}")
print(f"  Evidencia (respaldo):     {len(rag_evidence)}")
print(f"  Review (pendiente):       {len(rag_review_docs)}")
print(f"  Rejected (descartado):    {len(rag_rejected_docs)}")

In [ ]:
# Ejemplo de documento enriquecido
if rag_docs:
    sample = rag_docs[0]
    print("=" * 70)
    print("EJEMPLO: Documento enriquecido")
    print("=" * 70)
    print(f"doc_id:        {sample.doc_id}")
    print(f"source_type:   {sample.source_type}")
    print(f"topic:         {sample.topic}")
    print(f"keywords:      {sample.keywords}")
    print(f"slide_id:      {sample.slide_id}")
    print(f"timestamps:    {sample.start_ms}ms - {sample.end_ms}ms")
    print(f"quality_score: {sample.quality_score}")
    print(f"semantic_risk: {sample.semantic_risk}")
    print(f"linked_to:     {sample.linked_doc_id}")
    print(f"\n--- page_content ---")
    print(sample.page_content)
    
    # Evidencia enlazada
    linked = next((e for e in rag_evidence if e.doc_id == sample.linked_doc_id), None)
    if linked:
        print(f"\n{'=' * 70}")
        print("EJEMPLO: Documento evidencia (enlazado)")
        print("=" * 70)
        print(f"doc_id: {linked.doc_id}")
        print(f"\n--- page_content (primeros 300 chars) ---")
        print(linked.page_content[:300])

---
## 7. Agregar segmentos de audio como evidencia adicional

Para documentos que no tienen transcripción en el user_content,
buscamos el segmento de audio correspondiente por timestamps.

In [ ]:
def find_audio_segments(start_ms: int, end_ms: int, segments: List[Dict]) -> str:
    """Encuentra segmentos de audio que coincidan con el rango temporal."""
    matching = []
    for seg in segments:
        seg_start = seg.get("start_ms", 0)
        seg_end = seg.get("end_ms", 0)
        # Overlap check
        if seg_start < end_ms and seg_end > start_ms:
            matching.append(seg.get("text", "").strip())
    return " ".join(matching)


# Enriquecer evidencia con audio segments cuando esté vacía o corta
audio_segments = audio_data.get("segments", [])
enriched_count = 0

if audio_segments:
    for evidence_doc in rag_evidence:
        if len(evidence_doc.page_content.strip()) < 50:
            audio_text = find_audio_segments(evidence_doc.start_ms, evidence_doc.end_ms, audio_segments)
            if audio_text:
                evidence_doc.page_content = audio_text
                enriched_count += 1

print(f"Documentos evidencia enriquecidos con audio: {enriched_count}")

---
## 8. Guardar documentos RAG

In [ ]:
# Guardar
save_jsonl([d.to_dict() for d in rag_docs], RAG_DIR / "rag_docs.jsonl")
save_jsonl([d.to_dict() for d in rag_evidence], RAG_DIR / "rag_evidence.jsonl")
save_jsonl([d.to_dict() for d in rag_review_docs], RAG_DIR / "rag_review.jsonl")
save_jsonl([d.to_dict() for d in rag_rejected_docs], RAG_DIR / "rag_rejected.jsonl")

# Manifest
manifest = {
    "total_input": len(records),
    "docs_indexed": len(rag_docs),
    "evidence_indexed": len(rag_evidence),
    "review_pending": len(rag_review_docs),
    "rejected": len(rag_rejected_docs),
    "corrections_applied": total_corrections,
    "thresholds": {
        "good": THRESHOLD_GOOD,
        "review": THRESHOLD_REVIEW,
    },
    "quality_distribution": buckets,
    "canonical_terms_count": len(CANONICAL_TERMS),
}

with open(RAG_DIR / "rag_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f"Archivos guardados en {RAG_DIR}:")
for p in sorted(RAG_DIR.glob("*")):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:30s} {size_kb:8.1f} KB")

---
## 9. Resumen

In [ ]:
print("=" * 70)
print("NOTEBOOK 04: RAG DATA PREPARATION COMPLETADO")
print("=" * 70)
print(f"")
print(f"Input:  {len(records)} pseudo-labels")
print(f"")
print(f"Output:")
print(f"  rag_docs.jsonl      {len(rag_docs):4d} documentos enriquecidos (indice principal)")
print(f"  rag_evidence.jsonl  {len(rag_evidence):4d} documentos evidencia (respaldo)")
print(f"  rag_review.jsonl    {len(rag_review_docs):4d} pendientes de revision")
print(f"  rag_rejected.jsonl  {len(rag_rejected_docs):4d} descartados")
print(f"")
print(f"Correcciones canonicas: {total_corrections}")
print(f"")
print(f"Proximo paso: Notebook 05 (RAG Indexing)")
print("=" * 70)